In [2]:
import pandas as pd
import joblib

file_path = r"D:\S2\thesis\cond\project_ids\data\processed\split_balanced_data.pkl"

print("🔍 Mencoba buka gembok file...")

try:
    # Kunci 1: Pakai Pandas (Paling sering dipake buat nyimpen dataset)
    data = pd.read_pickle(file_path)
    print("✅ BINGO! Berhasil dibuka pakai Pandas.")
    
    # Cek isinya
    if isinstance(data, (list, tuple)):
        print("\nIsi filenya (List/Tuple):")
        for i, item in enumerate(data):
            print(f"Item {i} shape: {getattr(item, 'shape', 'Gak ada attribute shape')}")
    elif isinstance(data, dict):
        print("\nIsi filenya (Dictionary):")
        for key, value in data.items():
            print(f"{key} shape: {getattr(value, 'shape', 'Gak ada attribute shape')}")
    else:
        print(f"\nBentuk data: {type(data)}")
        print(f"Shape: {getattr(data, 'shape', 'Gak ada attribute shape')}")

except Exception as e:
    print(f"❌ Gagal pakai Pandas. Coba kunci 2...")
    
    try:
        # Kunci 2: Pakai Joblib
        data = joblib.load(file_path)
        print("✅ BINGO! Berhasil dibuka pakai Joblib.")
        print(f"Bentuk data: {type(data)}")
        if isinstance(data, (list, tuple)):
            for i, item in enumerate(data):
                 print(f"Item {i} shape: {getattr(item, 'shape', 'No shape')}")
    except Exception as e2:
        print("❌ Waduh, dua-duanya gagal. Berarti format simpannya beda lagi.")

🔍 Mencoba buka gembok file...
❌ Gagal pakai Pandas. Coba kunci 2...
✅ BINGO! Berhasil dibuka pakai Joblib.
Bentuk data: <class 'dict'>


In [3]:
import numpy as np
import torch

# Sekarang running lagi yang tadi:
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

print(f"✅ Data Terbaca dengan Aman!")
print(f"Input Fitur: {X_train.shape[1]}")
print(f"Jumlah Baris Training: {X_train.shape[0]}")
print(f"Jumlah Kelas: {len(np.unique(y_train))}")

✅ Data Terbaca dengan Aman!
Input Fitur: 44
Jumlah Baris Training: 1700000
Jumlah Kelas: 34


In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Konversi ke Tensor
X_original = torch.FloatTensor(X_train)
y_original = torch.LongTensor(y_train)

# 2. Tambahkan Gaussian Noise (samakan std dengan SLGRAE lo tadi)
# Kalau tadi lo pakai std=0.01, pakai itu lagi di sini
std_noise = 0.01 
noise = torch.randn_like(X_original) * std_noise
X_noised = X_original + noise

# 3. Concatenate (Gabungkan): Asli + Noised = 3,4 Juta
X_combined = torch.cat([X_original, X_noised], dim=0)
y_combined = torch.cat([y_original, y_original], dim=0)

print(f"✅ Dataset Tandingan Siap!")
print(f"Total Baris: {X_combined.shape[0]:,}") # Harus 3,400,000

# 4. Update DataLoader
train_ds_enet = TensorDataset(X_combined, y_combined)
train_loader_enet = DataLoader(train_ds_enet, batch_size=4096, shuffle=True)

✅ Dataset Tandingan Siap!
Total Baris: 3,400,000


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

# 1. Definisi Arsitektur Pure ENet (Tetap Sama)
class PureENet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(PureENet, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)

# 2. Setup Data & AUGMENTASI (CRITICAL!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("🔄 Menyiapkan data augmented (3,4 Juta Baris)...")
X_original = torch.FloatTensor(X_train)
y_original = torch.LongTensor(y_train)

# Tambahkan noise std=0.01 (samakan dengan SLGRAE lo)
noise = torch.randn_like(X_original) * 0.01
X_augmented = torch.cat([X_original, X_original + noise], dim=0)
y_augmented = torch.cat([y_original, y_original], dim=0)

input_dim = 44
num_classes = 34
epochs = 100 # Kasih 100 biar puas liat konvergensinya
batch_size = 4096 

train_ds = TensorDataset(X_augmented, y_augmented)
test_ds = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

model_enet = PureENet(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_enet.parameters(), lr=1e-3, weight_decay=1e-5)

scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.005, 
                                          steps_per_epoch=len(train_loader), 
                                          epochs=epochs)

# 3. Training Loop dengan Auto-Save
save_path = r"D:\S2\thesis\cond\project_ids\models\pure_enet_baseline.pth"
best_enet_acc = 0.0

print(f"🚀 Memulai Training Pure ENet Baseline (Adil - 3,4jt Baris)")

for epoch in range(epochs):
    model_enet.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_enet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    
    # Validasi
    model_enet.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_enet(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    epoch_loss = total_loss / len(train_loader)
    
    if acc > best_enet_acc:
        best_enet_acc = acc
        torch.save(model_enet.state_dict(), save_path)
        piala = "🏆 (SAVED)"
    else:
        piala = ""
    
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f} {piala}")

print(f"\n🏁 SELESAI! Best Pure ENet Accuracy: {best_enet_acc:.2f}%")

🔄 Menyiapkan data augmented (3,4 Juta Baris)...
🚀 Memulai Training Pure ENet Baseline (Adil - 3,4jt Baris)
Epoch [1/100] | Loss: 1.3278 | Acc: 78.88% | LR: 0.000213 🏆 (SAVED)
Epoch [2/100] | Loss: 0.9124 | Acc: 78.72% | LR: 0.000252 
Epoch [3/100] | Loss: 0.7849 | Acc: 78.33% | LR: 0.000317 
Epoch [4/100] | Loss: 0.6890 | Acc: 81.51% | LR: 0.000408 🏆 (SAVED)
Epoch [5/100] | Loss: 0.6306 | Acc: 80.98% | LR: 0.000522 
Epoch [6/100] | Loss: 0.5933 | Acc: 80.84% | LR: 0.000658 
Epoch [7/100] | Loss: 0.5711 | Acc: 83.88% | LR: 0.000816 🏆 (SAVED)
Epoch [8/100] | Loss: 0.5546 | Acc: 79.60% | LR: 0.000994 
Epoch [9/100] | Loss: 0.5436 | Acc: 82.90% | LR: 0.001189 
Epoch [10/100] | Loss: 0.5342 | Acc: 85.94% | LR: 0.001400 🏆 (SAVED)
Epoch [11/100] | Loss: 0.5325 | Acc: 78.39% | LR: 0.001624 
Epoch [12/100] | Loss: 0.5309 | Acc: 79.52% | LR: 0.001858 
Epoch [13/100] | Loss: 0.5284 | Acc: 82.79% | LR: 0.002101 
Epoch [14/100] | Loss: 0.5276 | Acc: 83.59% | LR: 0.002349 
Epoch [15/100] | Loss: 0.5

In [6]:
import torch
import os

# 1. Tentukan folder simpan (Samain dengan folder tesis lo)
save_directory = r"D:\S2\thesis\cond\project_ids\models"

if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# 2. Nama file sesuai rekor final malam ini
final_filename = "PureENet_Baseline_FINAL_96.27.pth"
save_path = os.path.join(save_directory, final_filename)

# 3. Proses simpan (State Dict & Metadata)
try:
    checkpoint = {
        'model_state_dict': model_enet.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_acc': 96.27,
        'final_loss': 0.3296,
        'epoch': 100,
        'notes': 'Pure ENet Baseline 44 fitur - Perbandingan Adil 3.4jt baris'
    }
    
    torch.save(checkpoint, save_path)
    print(f"✅ AMAN! Model Baseline 96.27% udah nangkring di: {save_path}")
    print("🚀 Sekarang lo bisa tidur tenang, PC udah bisa di-shutdown.")
except Exception as e:
    print(f"❌ WADUH GAGAL! Error: {e}")

✅ AMAN! Model Baseline 96.27% udah nangkring di: D:\S2\thesis\cond\project_ids\models\PureENet_Baseline_FINAL_96.27.pth
🚀 Sekarang lo bisa tidur tenang, PC udah bisa di-shutdown.
